In [22]:
"""
Using the already extracted daily bottom model-field
"""

'\nUsing the extracted daily bottom model-field\n\n/b1/yifan/LO_output/extract/cas2k_v4_lo2b/box/chrisfree\n\n'

In [23]:
import numpy as np
import xarray as xr
from lo_tools import Lfun, zrfun
from sklearn.preprocessing import StandardScaler
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import pandas as pd
import cmocean
Ldir = Lfun.Lstart()

In [24]:
# input location
type = 'box'
fod = 'chrisfree' # already extracted for this person
gtx = 'cas2k_v4_lo2b'
in_dir0 = Ldir['LOo'] / 'extract'/ gtx / type / fod
# output location
fod1 = 'bias_corrected'
out_dir0 = Ldir['LOo'] / 'extract'/ gtx / type / fod1
fig_dir0 = out_dir0 / 'figures'


In [25]:
# load trained model
dir = Ldir['LOo'] / 'LO_bias_corr'/'keras'
add_hake_hpic_trawl = True
cast_split = True
cast_balanced = True
use_z_fraction = True
benthic_aware = False
hypoxia_aware = False
hypoxia_shelf_aware = False
hypoxia_two_tier_shelf_aware = True
hypoxia_shelf_smooth_aware = False
remove_NO3 = True

if add_hake_hpic_trawl:
    runame = 'with_hake_hpic_trawl'
    if cast_split:
        name = 'cid_split'
        if cast_balanced:
            name += '_cast_balanced'
            if cast_balanced and use_z_fraction:
                name += '_z_fraction'
            if cast_balanced and benthic_aware:
                name += '_benthic_aware'
            if cast_balanced and hypoxia_aware:
                name += '_hypoxia_aware'
            if cast_balanced and hypoxia_shelf_aware:
                name += '_hypoxia_shelf_aware'
            if cast_balanced and hypoxia_shelf_smooth_aware:
                name += '_hypoxia_shelf_smooth_aware'
            if cast_balanced and hypoxia_two_tier_shelf_aware:
                name += '_hypoxia_two_tier_shelf_aware'
            if remove_NO3:
                name += '_no_NO3'

    else:
        name = ''

in_dir = dir/ runame / name
out_dir = out_dir0 / runame / name
fig_dir = fig_dir0 / runame / name

Lfun.make_dir(out_dir, clean=False)
Lfun.make_dir(fig_dir, clean=False)
# Load the saved model
model = load_model(f'{in_dir}/DO_bias_corr.keras')

# load scaler_x and y saved in training step

scaler_x = joblib.load(f'{in_dir}/scaler_x.pkl')
scaler_y = joblib.load(f'{in_dir}/scaler_y.pkl')

In [27]:
testing = False
if testing:
    year_list = [2021]
    # year_list = np.arange(1998,2001)
else:
    year_list = np.arange(1993,2022)

In [28]:
for year in year_list:
    print(year)
    fn = str(in_dir0/f'daily_bot_{year}_CO2SYS.nc')

    ds = xr.open_dataset(fn)

    lat = ds['lat_rho'].values
    lon = ds['lon_rho'].values

    # filter out salish sea
    """mask off Salish Sea"""

    filter = True

    if filter:
        # Define masks for regions to exclude (e.g., Puget Sound and Georgia Strait)
        filter = (

                ((lat > 45.0) & (lat < 50) & (lon >= -123.8) & (lon < -122.0))   # Puget Sound
                | ((lat > 48.9) & (lat < 51.0) & (lon > -125.5) & (lon < -123.7))    # Georgia Strait
                | ((lat > 50.5) & (lat < 51.0) & (lon > -126.5) & (lon < -125.5))
                | ((lat > 51.75) & (lat < 52.1) & (lon > -128.0) & (lon < -127.6))
                | ((lat >= 48.38) & (lat < 48.7) & (lon >= -124.65) & (lon < -123.725)) # JFS region
                | ((lat >= 48.15) & (lat < 48.7) & (lon >= -124.65) & (lon < -123.725)) # JFS region
                # | ((lat > 48.3)) # filter out northen BC coast
        )

        ds['mask_rho'].values[filter] = 0

    """Zoom into the shelf region"""
    ds = ds.where(
    (ds['mask_rho'] == 1) &
    (ds['h'] > 14.99) &
    (ds['h'] < 200.1)
    )

    # ds['h'].plot()
    # Read variables from the NetCDF file
    CT = ds['CT']
    SA = ds['SA']
    NO3 = ds['NO3']
    DO = ds['oxygen']
    z = ds['z_rho']
    h = -ds['h']
    z_fraction = (np.abs(z) / np.abs(h)).clip(min=0, max=1)

    time = ds['ocean_time']
    day_of_year = time.dt.dayofyear
    # Encode as sine and cosine
    sin_t = np.sin(2 * np.pi * day_of_year / 365)
    cos_t = np.cos(2 * np.pi * day_of_year / 365)

    lon = ds['lon_rho']   # (eta_rho, xi_rho)
    lat = ds['lat_rho']

    feat = xr.Dataset(
    data_vars=dict(
        lon      = lon,
        lat      = lat,
        z        = z,
        cos_t = cos_t,
        sin_t = sin_t,
        CT       = CT,
        SA       = SA,
        DO_uM    = DO,
        NO3_uM   = NO3,
        h = h,
        z_fraction = z_fraction,
    )
    )
    # broadcast lon/lat (and time features) to full (time,y,x)
    feat = feat.broadcast_like(CT)

    # --- stack into 1D samples ---
    feat_stacked = feat.stack(sample=('ocean_time',) + tuple([d for d in CT.dims if d != 'ocean_time']))

    # --- convert to a pandas table in the exact column order ---
    vn = [
        'lon',
        'lat',
        'z',
        'cos_t',
        'sin_t',
        'CT',
        'SA',
        'DO_uM',
        'NO3_uM',
    ]
    if remove_NO3:
        vn.remove('NO3_uM')
    if cast_balanced and use_z_fraction:
        vn.extend(['h', 'z_fraction'])

    X_pd = pd.DataFrame({k: feat_stacked[k].values for k in vn})

    # --- drop NaNs (important!) but keep indices to restore shape later ---
    valid = np.isfinite(X_pd).all(axis=1)
    X = X_pd.loc[valid]

    # --- scale X using the save scaler ---

    standardized_X = scaler_x.transform(X.values)
    #--- Predict---- plug into the bias-corr-model
    y_pred = model.predict(standardized_X)
    #---  scale back
    bias = scaler_y.inverse_transform(y_pred)
    """
    add the predicted bias back to modeled DO,
    """
    yy = X['DO_uM'] - bias.ravel()

    """add back to ds, using ds dimension"""
    # create empty full-length array
    corrected_all = np.full(X_pd.shape[0], np.nan)
    # fill only valid positions
    corrected_all[valid.values] = yy

    corrected_da = xr.DataArray(
    corrected_all,
    dims=('sample',),
    coords={'sample': feat_stacked['sample']}
    )

    # unstack back to original grid
    corrected_da = corrected_da.unstack('sample')

    ds['corrected_DO'] = corrected_da
    # print(ds['corrected_DO'].shape)
    # print(ds['oxygen'].shape)
    # Save the modified dataset back to the NetCDF file
    ds.to_netcdf(f"{out_dir}/daily_bot_{year}_CO2SYS.nc", mode='w', format='NETCDF4')
    print(f'bias_corrected DO data is saved for {year}')
    monthly = True
    if monthly:
        # average to monthly
        dt = ds.resample(ocean_time='ME').mean()
        Dir = out_dir / 'monthly'
        Lfun.make_dir(Dir)
        dt.to_netcdf(f'{Dir}/monthly_bot_{year}_CO2SYS.nc')
        dt.close()
    # Close the dataset
    ds.close()


1993


/home/yifan/anaconda3/envs/bias/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


92679/92679 ━━━━━━━━━━━━━━━━━━━━ 40s 433us/step
bias_corrected DO data is saved for 1993
1994


/home/yifan/anaconda3/envs/bias/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


93190/93190 ━━━━━━━━━━━━━━━━━━━━ 55s 589us/step
bias_corrected DO data is saved for 1994
1995


/home/yifan/anaconda3/envs/bias/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


... (49 outputs left out)

/home/yifan/anaconda3/envs/bias/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


93445/93445 ━━━━━━━━━━━━━━━━━━━━ 54s 578us/step
bias_corrected DO data is saved for 2020
2021


/home/yifan/anaconda3/envs/bias/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


93190/93190 ━━━━━━━━━━━━━━━━━━━━ 55s 593us/step
bias_corrected DO data is saved for 2021


In [29]:
if testing:
    ds['h'].min()
    ds['h'].max()
    ds['h'].plot()
    # ds['mask_rho'].plot()

In [33]:
if testing:
    i = 200
    date_str = str(ds['ocean_time'][i].values)[:10]  # YYYY-MM-DD
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

    # ---- Original DO ----
    im0 = ds['oxygen'][i,:,:].plot(
        ax=axes[0],
        vmin=0,
        vmax=300,
        cmap=cmocean.cm.oxy,
        add_colorbar=False
    )
    axes[0].set_title(f'Original DO,  {date_str}')

    # ---- Corrected DO ----
    im1 = ds['corrected_DO'][i,:,:].plot(
        ax=axes[1],
        vmin=0,
        vmax=300,
        cmap=cmocean.cm.oxy,
        add_colorbar=False
    )
    axes[1].set_title(f'Corrected DO,  {date_str}')

    # ---- Shared colorbar ----
    cbar = fig.colorbar(im1, ax=axes, orientation='vertical', shrink=0.9)
    cbar.set_label('(µM)')

    # ---- Save figure ----

    fig.savefig(f'{fig_dir}/DO_comparison_{date_str}.png', dpi=300, bbox_inches='tight')

    plt.show()

In [34]:
if testing:
    i = 8
    date_str = str(dt['ocean_time'][i].values)[:7]  # YYYY-MM-DD
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

    # ---- Original DO ----
    im0 = dt['oxygen'][i,:,:].plot(
        ax=axes[0],
        vmin=0,
        vmax=300,
        cmap=cmocean.cm.oxy,
        add_colorbar=False
    )
    axes[0].set_title(f'Original DO,  {date_str}')

    # ---- Corrected DO ----
    im1 = dt['corrected_DO'][i,:,:].plot(
        ax=axes[1],
        vmin=0,
        vmax=300,
        cmap=cmocean.cm.oxy,
        add_colorbar=False
    )
    axes[1].set_title(f'Corrected DO,  {date_str}')

    # ---- Shared colorbar ----
    cbar = fig.colorbar(im1, ax=axes, orientation='vertical', shrink=0.9)
    cbar.set_label('(µM)')

    # ---- Save figure ----

    fig.savefig(f'{fig_dir}/DO_comparison_monthly_mean_{date_str}.png', dpi=300, bbox_inches='tight')

    plt.show()

In [35]:
if testing:
    plt.figure(figsize=(8,4))
    plt.plot(time, sin_t, marker = '.', linestyle = '-', color = 'black', label='sin_time')
    plt.plot(time, cos_t, marker = '.', linestyle = '-', color = 'red',label='cos_time')
    plt.title('Seasonal Encoding with Sine and Cosine')
    plt.xlabel('Date')
    plt.ylabel('Encoded Value')
    plt.legend()
    plt.grid(True)
    # plt.tight_layout()
    plt.show()